# 4. Working with sequences

Much of tcrstruct is sequence work: finding the chains, numbering them, and matching a
model's chains to the reference before scoring. This notebook does that work on sequences
alone, with no coordinates. The alignments are PDBFrame's `SeqFrame`
([PDBFrame docs/seqframe.md](../../PDBFrame/docs/seqframe.md)).

In [1]:
import pandas as pd
from pdbframe import PDBFrame, SeqFrame

import tcrstruct as tcs
from tcrstruct import TCRFrame

pd.options.display.precision = 3

## Sequences only

`TCRFrame.from_sequences` takes `{chain: sequence}` and annotates it as it would a
structure: chain roles, genes, IMGT numbering and CDRs. Here are the deposited
sequences of 1AO7: an mmCIF file lists them whole, residues without atoms included
(`full_sequences`; `sequences` has only the residues with atoms):

In [2]:
pf = PDBFrame.from_rcsb('1AO7')        # the mmCIF file, the file's chain IDs
full = pf.full_sequences
deposited = TCRFrame.from_sequences(full, name='1AO7 deposited')
print(deposited.report())

TCRFrame '1AO7 deposited', MHC class I, chains A, C, D, E, F
  A  MHC heavy chain A0201        275 aa   (chain A in the file)
  C  peptide                        9 aa   (chain C in the file)
  D  TCR alpha       TRAV12-2*01  204 aa   (chain D in the file)
  E  TCR beta        TRBV6-5*01   245 aa   (chain E in the file)
  F  B2M             B2M_human    100 aa   (chain B in the file)
  peptide LLFGYPVYV
  CDRs    CDRA1=DRGSQS  CDRA2=IYSNGD  CDRA3=AVTTDSWGKLQ  CDRB1=MNHEY  CDRB2=SVGAGI  CDRB3=ASRPGLAGGRPEQY


Per chain: the MHC gene and allele with the identity to that allele, and for the TCR
chains the V gene and allele with ANARCI's identity to it, then the J allele and its
identity:

In [3]:
deposited.chain_table.drop(columns='sequence')

,role,gene,allele,identity,j_gene,j_identity,original_chain,length
chain,,,,,,,,
A,MHC heavy chain,A,A0201,1.000,-,NaN,A,275
C,peptide,-,-,NaN,-,NaN,C,9
D,TCR alpha,TRAV12-2,TRAV12-2*01,0.957,TRAJ24*02,1.0,D,204
E,TCR beta,TRBV6-5,TRBV6-5*01,0.979,TRBJ2-7*01,1.0,E,245
F,B2M,B2M,B2M_human,0.980,-,NaN,B,100


`tcs.detect_chains` is the step behind that. It shows what each input chain was taken
to be:

In [4]:
tcs.detect_chains(full)

,type,details,new_chain,sequence
A,MHC,A,A,GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...
B,MHC,B2M,F,MIQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGER...
C,peptide,-,C,LLFGYPVYV
D,TCR,Alpha,D,KEVEQNSGPLSVPEGAIASLNCTYSDRGSQSFFWYRQYSGKSPELI...
E,TCR,Beta,E,NAGVTQTPKFQVLKTGQSMTLQCAQDMNHEYMSWYRQDPGMGLRLI...


## Deposited vs modelled residues

A crystal structure often lacks residues the protein has. Aligning the TCR alpha's
deposited sequence (`seqframe(..., full=True)`) to its residues with atoms shows that most
of its constant domain was not resolved; those residues are in `structure.unobserved`:

In [5]:
structure = TCRFrame(pf, name='1AO7')
aln = structure.seqframe('D', full=True).align(structure.seqframe('D'),
                                               names=['deposited', 'structure'])
aln.show()
structure.unobserved.groupby('chain').size()

deposited     1 KEVEQNSGPLSVPEGAIASLNCTYSDRGSQSFFWYRQYSGKSPELIMSIYSNGDKEDGRF 60
                ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
structure     1 KEVEQNSGPLSVPEGAIASLNCTYSDRGSQSFFWYRQYSGKSPELIMSIYSNGDKEDGRF 60

deposited    61 TAQLNKASQYVSLLIRDSQPSDSATYLCAVTTDSWGKLQFGAGTQVVVTPDIQNPDPAVY 120
                |||||||||||||||||||||||||||||||||||||||||||||||||||||||     
structure    61 TAQLNKASQYVSLLIRDSQPSDSATYLCAVTTDSWGKLQFGAGTQVVVTPDIQNP----- 115

deposited   121 QLRDSKSSDKSVCLFTDFDSQTNVSQSKDSDVYITDKTVLDMRSMDFKSNSAVAWSNKSD 180
                                                                            
structure       ------------------------------------------------------------ 

deposited   181 FACANAFNNSIIPEDTFFPSPESS 204
                                        
structure       ------------------------ 



chain
A     1
D    89
E    36
dtype: int64

## Model vs reference: what gets scored

`compare` aligns each model chain to the reference chain with the same ID and keeps
only the identical residues. A chain is skipped if its identity is below `min_identity`
(0.3). The identity counts every alignment column, gaps included, so a TCRmodel2 model
(V domains only) scores about 0.47 on the 3QIB TCR beta chain:

In [6]:
reference = TCRFrame.read(tcs.example('3QIB', 'rcsb.pdb'), name='3QIB')
tcrmodel2 = TCRFrame.read(tcs.example('3QIB', 'tcrmodel2.pdb'), name='TCRmodel2')
aln = reference.seqframe('E').align(tcrmodel2.seqframe('E'), names=['reference', 'TCRmodel2'])
aln.show()
aln.statistics[['identity', 'matches', 'gap_columns']]

reference     1 -MKVIQTPRYLVKGQGQKAKMRCIPEKGHPVVFWYQQNKNNEFKFLINFQNQEVLQQIDM 59
                 |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
TCRmodel2     1 GMKVIQTPRYLVKGQGQKAKMRCIPEKGHPVVFWYQQNKNNEFKFLINFQNQEVLQQIDM 60

reference    60 TEKRFSAECPSNSPCSLEIQSSEAGDSALYLCASSLNWSQDTQYFGPGTRLLVLEDLKNV 119
                |||||||||||||||||||||||||||||||||||||||||||||||||||||||     
TCRmodel2    61 TEKRFSAECPSNSPCSLEIQSSEAGDSALYLCASSLNWSQDTQYFGPGTRLLVLE----- 115

reference   120 FPPEVAVFEPSEAEISHTQKATLVCLATGFYPDHVELSWWVNGKEVHSGVCTDPQPLKEQ 179
                                                                            
TCRmodel2       ------------------------------------------------------------ 

reference   180 PALNDSRYALSSRLRVSATFWQNPRNHFRCQVQFYGLSENDEWTQDRAKPVTQIVSAEAW 239
                                                                            
TCRmodel2       ------------------------------------------------------------ 

reference   240 GRA 242
                   
TCRm

identity       0.469
matches          114
gap_columns      129
Name: statistics, dtype: object

The identities `compare` uses, per chain, are the same numbers (`tcs.metrics.match`, the
first step of `compare`):

In [7]:
_, _, identity = tcs.metrics.match(reference, tcrmodel2)
pd.Series(identity)

A    0.450
B    0.545
C    1.000
D    0.556
E    0.469
dtype: float64

## Comparing TCRs by their CDRs

The MHC allele, peptide, V genes and CDRs of the complexes in the test data, from each
one's `summary()`, and their CDR3β loops aligned to the first one:

In [8]:
complexes = {p: TCRFrame.read(tcs.example(p, 'rcsb.pdb'), name=p)
             for p in ['1AO7', '6ZKW', '3QIB', '7T2D', '7N2N']}
summary = pd.DataFrame({p: s.summary() for p, s in complexes.items()}).T
summary[['mhc', 'peptide', 'TRAV', 'TRBV', *tcs.regions.CDRS]]

,mhc,peptide,TRAV,TRBV,CDRA1,CDRA2,CDRA3,CDRB1,CDRB2,CDRB3
1AO7,A0201,LLFGYPVYV,TRAV12-2*01,TRBV6-5*01,DRGSQS,IYSNGD,AVTTDSWGKLQ,MNHEY,SVGAGI,ASRPGLAGGRPEQY
6ZKW,E0103,RLPAKAPLL,TRAV21*01,TRBV6-5*01,DSAIYN,IQSSQRE,AVTNQAGTALI,MNHEY,SVGAGI,ASSYSIRGSRGEQF
3QIB,H2-EAk,LIAYLKQATKG,TRAV4-4/DV10*02,TRBV26*01,TTMRA,LASGT,AALRATGGNNKLT,KGHPV,FQNQEV,ASSLNWSQDTQY
7T2D,DPA1*0103,GLAWEWWRTVY,TRAV19*01,TRBV7-3*01,TRDTTYY,RNSFDEQN,ALSGSARQLT,SGHTA,FQGTGA,ASSHREGETQY
7N2N,B2705,TRLALIAPK,TRAV21*01,TRBV9*01,DSAIYN,IQSSQRE,AVLSPVQETSGSRLT,SGDLS,YYNGEE,ASSVGLFSTDTQY


In [9]:
cdr3b = SeqFrame.align(*[s.cdrs['CDRB3'] for s in complexes.values()], names=list(complexes))
cdr3b.show(reference=True)
pd.DataFrame({n: p.statistics for n, p in cdr3b.pairwise().items()}).loc[['identity', 'similarity']]

1AO7     1 AS--RPGLAGGRPE-QY 14
6ZKW     1 ..--SYSIR.S.G.-.F 14
3QIB     1 ..----S.NWSQDT-.. 12
7T2D     1 ..SH.E.------.T.. 11
7N2N     1 ..--SV..-FSTDT-.. 13
           ||    .      . |:



,6ZKW,3QIB,7T2D,7N2N
identity,0.429,0.357,0.412,0.429
similarity,0.571,0.357,0.412,0.429
